# THỰC HÀNH 4 (LAB 04): DỰ BÁO CHUỖI THỜI GIAN ĐA BIẾN (MULTIVARIATE TIME SERIES FORECASTING)
**Học phần:** Phân tích Chuỗi Thời gian (Time Series Analysis — UTH)  
- **Bài toán 1 (Yêu cầu 1–11):** Dự báo phụ tải điện & thời tiết (`data/raw/electricity_load.csv`) với **VAR**, **XGBoost** và **LSTM** (`LSTM(64) -> LSTM(32) -> Dense(1)`).
- **Bài toán 2 (Yêu cầu 12 Nâng cao):** Dự báo tốc độ giao thông đa bước 15, 30, 60 phút (`data/raw/pems-bay.h5` + `data/raw/sensor_graph/adj_mx_bay.pkl`) với **DCRNN (DCGRU Seq2Seq)**.
- **Kiểm định & Đánh giá:** Chia Chronological `70% Train - 15% Val - 15% Test (shuffle=False)` kết hợp `5-Fold TimeSeriesSplit` và `Walk-Forward Backtesting` trên bộ 5 chỉ số: **MSE, RMSE, MAE, MAPE (%), R²**.


## Step 1: Khởi tạo Môi trường, Cố định Seed & Tự động Nhận diện GPU (`Auto-GPU Dispatcher`)


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().resolve()
while not (ROOT_DIR / 'src').exists() and ROOT_DIR.parent != ROOT_DIR:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.lab04.utils import load_config, set_global_seeds, detect_hardware_accelerator
from src.lab04.tracking import create_run

cfg = load_config()
seed_info = set_global_seeds(cfg['project']['random_state'])
hw_info = detect_hardware_accelerator(cfg)
run_id = create_run(metadata={'seed_info': seed_info, 'hardware': hw_info})
print(f'Run ID       : {run_id}')
print(f'Compute Dev  : {hw_info["device_name"]} (torch={hw_info["torch_device"]}, xgb={hw_info["xgb_device"]}, AMP_FP16={hw_info["amp_enabled"]})')


## Step 2: Nạp Bộ Dữ liệu Kép (`electricity_load.csv`, `pems-bay.h5` & `sensor_graph/adj_mx_bay.pkl`)


In [ ]:
from src.lab04 import data_loader
data_summary = data_loader.main(run_id)
df_elec, elec_meta = data_loader.load_electricity_dataset(cfg)
display(df_elec.head(10))


## Step 3: Khám phá Dữ liệu (`Data Understanding`: Missing Values, Outliers, Seasonality & Tương quan)


In [ ]:
from src.lab04 import eda
eda_summary = eda.main(run_id)
stats_df = pd.read_csv(ROOT_DIR / cfg['paths']['metrics_dir'] / 'eda_descriptive_statistics.csv')
corr_df = pd.read_csv(ROOT_DIR / cfg['paths']['metrics_dir'] / 'eda_correlation_matrix.csv', index_col=0)
display(stats_df)
display(corr_df)


## Step 4 & 5: Tiền xử lý (`StandardScaler`), Chia `70/15/15`, `5-Fold TimeSeriesSplit`, Kiểm định ADF & Tạo Đặc trưng Lag/Rolling


In [ ]:
from src.lab04 import preprocess
prep_summary = preprocess.main(run_id)
print('Chronological Split Counts:', prep_summary['chronological_split_counts'])
print('Supervised Window Shapes  :', prep_summary['supervised_window_shapes'])
print('ADF Stationarity Report   :', prep_summary['adf_stationarity_summary']['diff_order_applied'])


## Step 6, 7, 8 & 12: Huấn luyện Đa Mô hình (`VAR`, `XGBoost`, `LSTM 64->32->1` và `DCRNN` trên `Auto-GPU`)


In [ ]:
from src.lab04 import train
train_summary = train.main(run_id)
df_imp = pd.read_csv(ROOT_DIR / cfg['paths']['metrics_dir'] / 'xgboost_feature_importance.csv')
display(df_imp.head(15))


## Step 9, 10 & 11: Walk-Forward Backtesting, Kiểm định 5-Fold CV & Đánh giá 5 Chỉ số (`MSE`, `RMSE`, `MAE`, `MAPE (%)`, `R²`)


In [ ]:
from src.lab04 import evaluate
eval_summary = evaluate.main(run_id)
df_test_metrics = pd.read_csv(ROOT_DIR / cfg['paths']['metrics_dir'] / 'test_metrics_comparison.csv')
df_kfold_metrics = pd.read_csv(ROOT_DIR / cfg['paths']['metrics_dir'] / 'kfold_cv_results.csv')
print('=== KẾT QUẢ TRÊN TẬP HOLD-OUT TEST (15%) ===')
display(df_test_metrics)
print('=== KẾT QUẢ TRÊN 5-FOLD TIMESERIESSPLIT ===')
display(df_kfold_metrics)


## Step 12: Kết quả Dự báo Giao thông Đa bước (15, 30, 60 Phút) với `DCRNN` trên `PEMS-BAY` & Xuất Báo cáo Tự động (`.pdf`, `.html`, `.md`)


In [ ]:
from src.lab04 import report
pems_csv = ROOT_DIR / cfg['paths']['metrics_dir'] / 'pems_bay_metrics.csv'
if pems_csv.exists():
    display(pd.read_csv(pems_csv))

for q_k, q_ans in eval_summary['section11_analytical_answers'].items():
    print(f'[{q_k}]: {q_ans}\n')

report_info = report.main(run_id)
print('Đã tự động kết xuất báo cáo:', report_info)
